# ARC Canon-AC end-to-end smoke

This is a bounded mechanical gate, not a score experiment. It uses the real four-L4 global-training stack on 8 public-evaluation tasks x 4 views, trains zero-initialized residual Canon-AC alone for the first 30% of records and Canon plus a fresh rank-256 global LoRA for the rest, merges the ordinary adapters, then runs vanilla 128-step TTFT and one-token DFS on one validation puzzle. It verifies the released-paper boundary semantics, Canon weight movement, sidecar reload, branch-local incremental state, valid candidate production, and validation scoring. No speculative decoding is used in this first gate.


In [ ]:
from pathlib import Path

EXPECTED_SOURCE_HASHES = {'arc_canon.py': '6a91b637f3776d431ea7f615e35f50ba88094b8cc6ed7180148b194c698c1356', 'test_arc_canon.py': 'd3e7d3d4c2fb2205fe7112f4e2d6d66dc95f89855ec9e4a754133ef2a889f868', 'arc_search.py': '3444edcbeebafcb4c3f8f8a8e353511c96fbbf177e0f8b0aa4721c5e1fd5762d', 'arc_solver.py': '7cf1245cc9033055aaa3efb381af95ae8cdc71dbcd2c5322b15878dd9978140e', 'starter.py': '73f34783ee41c7c04b248d26f97dd95ae7ec2fda32faeb0fa9d3afa22f871074', 'global_eval_curriculum.py': '9764f4a397ca79e5fbb6f9042831911bfd452981772b23ec8a0307a06d050c27', 'train_global_eval_curriculum.py': '6250a89551fb8484f3c2ff30543405fc713239ea4ad4d40ce208ec39407e2eca', 'arc_loader.py': 'f4471da5448c8117ec48647d6c91f386fa892894bd5b5781e610d04518e61d44', 'arc_decoder.py': '2e76ebee24466986f6bb50054f0e55e2dfd055731f235139bc8b5444c2448157', 'arc_rescoring.py': '558b43ec0b7886944076c4ce9e506dc734ab123b0e6870d20706c4ac46afe5bc', 'arc_opsd.py': '7e9c0ecdc3328ea61c1c2daca2a54ba247f32dc974ddcfc503541efadfa6228c', 'arc_repair_ttft.py': 'fbc51d338d474e1857e9df046cc8a30933169ab1e98cac113424d3d26b7b2642', 'arc_selected_augmentations.py': '8f6ae666a4562e3c6386bd5b8d51a80009b0afc284a70c15c75d71d447fea7eb', 'arc_scheduled_sampling.py': 'd4b05a85a58f0b8db7dc389bba50ad66ee28f39cb0f5a861e4e26f924b3c0ee1', 'arc_sglang.py': '840b7af0d71bec3c5231a0865019f76a5d9ce7c063a232ec0410c0990a84ad3e', 'train_repair_adapter.py': 'dbc5dd41e96d924362371b2c87d26d17e1125de68e422d21c195e06e1cd7bb5b', 'repair_mining.py': 'c67d444662267b6eaeff5dc93b65f33cdc0b0dc7de1e02aa5b71f5d5d1f27c28', 'repair_sft.py': 'c044b97afbca2f69199901103bc98d96287793485a5384eb4056071bdf532fcc'}
MODEL_PATH = Path('/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1')
COMP_ROOT = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
CHALLENGES_PATH = COMP_ROOT / 'arc-agi_evaluation_challenges.json'
SOLUTIONS_PATH = COMP_ROOT / 'arc-agi_evaluation_solutions.json'
SMOKE_KEY = '0934a4d8'

WORK_ROOT = Path('/kaggle/working/arc26_canon_ac_smoke')
WORK_CODE_DIR = WORK_ROOT / 'ARC-AGI1/qwen_baseline'
GLOBAL_ROOT = Path('/kaggle/working/canon_global_smoke')
GLOBAL_ADAPTER = GLOBAL_ROOT / 'adapter'
MERGED_MODEL = Path('/kaggle/working/canon_merged_smoke')
INFERENCE_OUTPUT = Path('/kaggle/working/canon_smoke_candidates')

MODERN_UTILITY_ROOT = Path('/kaggle/usr/lib/notebooks/yuvraj/pip_install_unsloth_ddp_repair')
print('source files =', len(EXPECTED_SOURCE_HASHES))


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys

gpu_names = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
).strip().splitlines()
if len(gpu_names) != 4 or any('L4' not in name for name in gpu_names):
    raise RuntimeError(f'Expected four L4 GPUs, observed {gpu_names}')

code_candidates = [
    Path('/kaggle/input/datasets/yuvraj/arc2026'),
    Path('/kaggle/input/arc2026'),
]
CODE_DATASET_ROOT = next((path for path in code_candidates if path.is_dir()), None)
if CODE_DATASET_ROOT is None:
    raise FileNotFoundError(code_candidates)
source_root = CODE_DATASET_ROOT / 'ARC-AGI1/qwen_baseline'
observed = {
    name: hashlib.sha256((source_root / name).read_bytes()).hexdigest()
    for name in EXPECTED_SOURCE_HASHES
}
if observed != EXPECTED_SOURCE_HASHES:
    raise RuntimeError(f'arc2026 source mismatch: expected={EXPECTED_SOURCE_HASHES} observed={observed}')

repair_candidates = [
    Path('/kaggle/input/notebooks/yuvraj/arc26-repair-sft-continuation/repair_sft_continued/adapter'),
    Path('/kaggle/input/arc26-repair-sft-continuation/repair_sft_continued/adapter'),
]
REPAIR_ADAPTER = next((path for path in repair_candidates if path.is_dir()), None)
if REPAIR_ADAPTER is None:
    raise FileNotFoundError(repair_candidates)

fa2_candidates = sorted(set(
    path.parent.parent
    for root in (Path('/kaggle/input'), Path('/kaggle/usr/lib/notebooks'))
    if root.exists()
    for path in root.glob('**/flash_attn/__init__.py')
))
if len(fa2_candidates) != 1:
    raise RuntimeError(f'Expected one FlashAttention root, found {fa2_candidates}')
FA2_ROOT = fa2_candidates[0]

for path in (WORK_ROOT, GLOBAL_ROOT, MERGED_MODEL, INFERENCE_OUTPUT):
    shutil.rmtree(path, ignore_errors=True)
shutil.copytree(CODE_DATASET_ROOT, WORK_ROOT)
INFERENCE_OUTPUT.mkdir(parents=True)

environment = os.environ.copy()
environment.update({
    'UNSLOTH_DISABLE_STATISTICS': '1',
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'HF_HUB_ENABLE_HF_TRANSFER': '0',
    'TRITON_PTXAS_PATH': '/usr/local/cuda/bin/ptxas',
    'OMP_NUM_THREADS': '3',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
old_parts = [
    part for part in environment.get('PYTHONPATH', '').split(os.pathsep)
    if part and 'pip_install_unsloth_' not in part and 'flash_attention_' not in part
]
environment['PYTHONPATH'] = os.pathsep.join([
    str(FA2_ROOT), str(MODERN_UTILITY_ROOT), '/kaggle/working', str(WORK_CODE_DIR), *old_parts
])
print('GPUs =', gpu_names)
print('code =', CODE_DATASET_ROOT)
print('repair =', REPAIR_ADAPTER)
print('utility =', MODERN_UTILITY_ROOT)
print('FA2 =', FA2_ROOT)


In [ ]:
# Cheap operator tests run before model loading.
subprocess.run(
    [sys.executable, '-m', 'unittest', 'test_arc_canon.py'],
    cwd=WORK_CODE_DIR,
    env=environment,
    check=True,
)


In [ ]:
import time

train_command = [
    sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node', '4',
    str(WORK_CODE_DIR / 'train_global_eval_curriculum.py'),
    '--model-path', str(MODEL_PATH),
    '--repair-adapter-path', str(REPAIR_ADAPTER),
    '--challenges-path', str(CHALLENGES_PATH),
    '--output-dir', str(GLOBAL_ROOT),
    '--max-tasks', '8',
    '--views-per-task', '4',
    '--score-batch-size', '2',
    '--epochs', '1.0',
    '--learning-rate', '2.5e-5',
    '--lora-rank', '256',
    '--gradient-accumulation-steps', '1',
    '--expected-world-size', '4',
    '--canon-ac',
    '--canon-kernel-size', '4',
    '--canon-only-warmup-fraction', '0.30',
]
started = time.perf_counter()
print('running:', ' '.join(train_command))
subprocess.run(train_command, env=environment, check=True)
global_seconds = time.perf_counter() - started
manifest = json.loads((GLOBAL_ROOT / 'global_eval_curriculum_manifest.json').read_text())
print('global seconds =', round(global_seconds, 2))
print('stages =', json.dumps(manifest['train_stages'], indent=2))
print('canon l2 =', manifest['model']['canon_l2_before'], '->', manifest['model']['canon_l2_after'])
for required in ('adapter_model.safetensors', 'adapter_config.json', 'canon_ac.pt'):
    if not (GLOBAL_ADAPTER / required).is_file():
        raise FileNotFoundError(GLOBAL_ADAPTER / required)


In [ ]:
import textwrap

merge_code = textwrap.dedent(r'''import json
import os
import shutil
from pathlib import Path

import unsloth
from peft import PeftModel
from unsloth import FastLanguageModel

model_path = os.environ['ARC_BASE_MODEL_PATH']
repair_path = Path(os.environ['ARC_REPAIR_ADAPTER_PATH'])
global_path = Path(os.environ['ARC_GLOBAL_ADAPTER_PATH'])
output_path = Path(os.environ['ARC_MERGED_MODEL_PATH'])
supported = {
    'base_model_name_or_path', 'bias', 'fan_in_fan_out', 'inference_mode',
    'init_lora_weights', 'layers_pattern', 'layers_to_transform', 'loftq_config',
    'lora_alpha', 'lora_dropout', 'megatron_config', 'megatron_core',
    'modules_to_save', 'peft_type', 'r', 'rank_pattern', 'revision',
    'target_modules', 'task_type', 'use_dora', 'use_rslora',
}

def compatible_adapter(source, name):
    destination = Path('/kaggle/working') / name
    shutil.rmtree(destination, ignore_errors=True)
    destination.mkdir(parents=True)
    raw = json.loads((source / 'adapter_config.json').read_text())
    config = {key: value for key, value in raw.items() if key in supported}
    config['base_model_name_or_path'] = model_path
    (destination / 'adapter_config.json').write_text(json.dumps(config, indent=2) + '
')
    os.symlink(source / 'adapter_model.safetensors', destination / 'adapter_model.safetensors')
    return destination

repair_compat = compatible_adapter(repair_path, 'canon_repair_compat')
global_compat = compatible_adapter(global_path, 'canon_global_compat')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path, full_finetuning=False, load_in_4bit=False,
    local_files_only=True, use_gradient_checkpointing=False, max_seq_length=8192,
)
existing = getattr(tokenizer, 'additional_special_tokens', None)
if existing is None:
    existing = (getattr(tokenizer, 'special_tokens_map', {}) or {}).get('additional_special_tokens', [])
existing = list(existing)
if '<REPAIR>' not in existing:
    existing.append('<REPAIR>')
    if tokenizer.add_special_tokens({'additional_special_tokens': existing}) != 1:
        raise RuntimeError('Failed to add exactly one repair token')
try:
    model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
except TypeError:
    model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(model, str(repair_compat), is_trainable=False, local_files_only=True)
model = model.merge_and_unload(safe_merge=True)
model = PeftModel.from_pretrained(model, str(global_compat), is_trainable=False, local_files_only=True)
model = model.merge_and_unload(safe_merge=True)
output_path.mkdir(parents=True)
model.save_pretrained(output_path, safe_serialization=True, max_shard_size='5GB')
tokenizer.save_pretrained(output_path)
print('saved merged ordinary weights:', output_path)
''')

merge_env = environment.copy()
merge_env.update({
    'ARC_BASE_MODEL_PATH': str(MODEL_PATH),
    'ARC_REPAIR_ADAPTER_PATH': str(REPAIR_ADAPTER),
    'ARC_GLOBAL_ADAPTER_PATH': str(GLOBAL_ADAPTER),
    'ARC_MERGED_MODEL_PATH': str(MERGED_MODEL),
})
subprocess.run([sys.executable, '-c', merge_code], env=merge_env, check=True)


In [ ]:
import time

inference_command = [
    sys.executable, str(WORK_CODE_DIR / 'starter.py'),
    '--test-path', str(CHALLENGES_PATH),
    '--model-path', str(MERGED_MODEL),
    '--output-dir', str(INFERENCE_OUTPUT),
    '--keys-json', json.dumps([SMOKE_KEY]),
    '--nprocs', '1',
    '--dfs-prob-threshold', '0.2',
    '--eval-color-permutations', '2',
    '--ttft-method', 'full_sft',
    '--canon-ac-state', str(GLOBAL_ADAPTER / 'canon_ac.pt'),
    '--end-time', str(time.time() + 45 * 60),
    '--profile-timings',
]
started = time.perf_counter()
print('running:', ' '.join(inference_command))
subprocess.run(inference_command, cwd=WORK_CODE_DIR, env=environment, check=True)
inference_seconds = time.perf_counter() - started
print('inference seconds =', round(inference_seconds, 2))
print('candidate files =', len([path for path in INFERENCE_OUTPUT.iterdir() if path.is_file()]))


In [ ]:
import bz2
import pickle

import numpy as np

sys.path.insert(0, str(WORK_CODE_DIR))
from arc_decoder import ArcDecoder, score_kgmon
from arc_loader import ArcDataset

data = ArcDataset.from_file(CHALLENGES_PATH, keys=[SMOKE_KEY]).load_replies(SOLUTIONS_PATH)
split_data = data.split_multi_replies()
decoder = ArcDecoder(split_data, n_guesses=2)
candidate_records = 0
for path in sorted(INFERENCE_OUTPUT.iterdir()):
    if not path.is_file():
        continue
    with bz2.BZ2File(path, 'rb') as handle:
        rows = pickle.load(handle)
    base_key = path.name.split('.')[0]
    decoder.decoded_results.setdefault(base_key, {})
    for index, row in enumerate(rows):
        decoder.decoded_results[base_key][f'{path.name}.out{index}'] = row
        candidate_records += 1

selected = decoder.run_selection_algo(score_kgmon)
submission = data.get_submission(selected)
score = data.validate_submission(submission)
summary = {
    'key': SMOKE_KEY,
    'candidate_files': len([path for path in INFERENCE_OUTPUT.iterdir() if path.is_file()]),
    'candidate_records': candidate_records,
    'decoded_outputs': len(decoder.decoded_results),
    'selected_score': score,
    'global_seconds': global_seconds,
    'inference_seconds': inference_seconds,
    'canon_l2_before': manifest['model']['canon_l2_before'],
    'canon_l2_after': manifest['model']['canon_l2_after'],
}
(Path('/kaggle/working') / 'canon_ac_smoke_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
